In [1]:
import torch 
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters 
batch_size = 32         # No of sequences processed in parallel during the training 
block_size = 8          # Maximum context length (how many tokens the model looks back)
max_iters = 3000        # Total no of traning interations 
eval_interval = 300     # How often (in iterations) we run evalution (0.01)
learning_rate = 1e-2    # Step size for updating the model parameters (controls speed of learning)
device = 'cuda' if torch.cuda.is_available() else 'cpu' # set the device to use 
eval_iters=200          # No of iterations used to average evaluation metrics 

In [2]:
torch.manual_seed(1337) # sets the starting point (seed) for the internal pseudo-random number generator(RNG)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
# here are the all the unique charaters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)     # vocabulary size is the total count of unique tokens (words, subwords, characters) the model recognizes and uses for processing text
print(chars)
vocab_size 

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


65

In [4]:
# Create mapping from charaters to integers

stoi = { ch: i for i, ch in enumerate(chars)}    # string to integer
itos = { i: ch for i, ch in enumerate(chars)}    # integer to string
encode = lambda s:[stoi[c] for c in s]   # encoder: take a string, output a list of integers
decode = lambda l:  ''.join([itos[i]for i in l]) # decoder: take a list of integers, output a string

In [5]:
# train and test splits 

data = torch.tensor(encode(text), dtype=torch.long)     # torch.tensor(), creates a multi-dimensional array (tensor) from existing Python data (like lists or tuples).
n = int(0.9*len(data))     # first 90% train set and rest is val
train_data = data[:n]
val_data = data[n:]

In [6]:
# data loading 

def get_batch(split):
    # choose training data or validation data
    data = train_data if split == 'train' else val_data 
    ix = torch.randint(len(data) - block_size, (batch_size,))   # pick random starting positions for each sequence in the batch
    x = torch.stack([data[i:i+block_size] for i in ix])         # create input sequences of length block_size
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])     # create target sequences (shifted by one position)
    x, y = x.to(device), y.to(device)     # move tensors to GPU if available, else CPU
    return x, y                           # return input and target batch

In [7]:
# super simple bigram model 
class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # embedding table: maps token → logits for next token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets=None):
        # idx, targets: (B, T) tensors of token indices
        logits = self.token_embedding_table(idx)    # (B, T, C) predictions
        
        if targets is None:
            loss = None
        else:  
            B, T, C = logits.shape                  # batch, time, vocab size
            logits = logits.view(B*T, C)            # flatten for loss
            targets = targets.view(B*T)             # flatten targets
            loss = F.cross_entropy(logits, targets) # compute loss
            
        return logits, loss
            
    def generate(self, idx, max_new_tokens):
        # idx: (B, T) current context
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                   # forward pass
            logits = logits[:, -1, :]               # last time step (B, C)
            probs = F.softmax(logits, dim=-1)       # convert to probabilities
            idx_next = torch.multinomial(probs, 1)  # sample next token (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # append to sequence (B, T+1)
            
        return idx                                  # final generated sequence


In [8]:
# create an instance of the BigramLanguageModel with given vocabulary size
model = BigramLanguageModel(vocab_size)

# move the model to the chosen device (CPU or GPU)
m = model.to(device)

In [9]:
@torch.no_grad()                     # disable gradient tracking (faster, less memory)
def estimate_loss():
    out = {}                         # dictionary to store average losses
    model.eval()                     # set model to evaluation mode (no dropout, etc.)
    for split in ['train', 'val']:  # loop over both train and eval datasets
        losses = torch.zeros(eval_iters)   # store losses for multiple iterations
        for k in range(eval_iters):        # run eval_iters times
            X, Y = get_batch(split)        # get a batch of data
            logits, loss = model(X, Y)     # forward pass, compute loss
            losses[k] = loss.item()        # save loss value
        out[split] = losses.mean()         # average loss for this split
    model.train()                          # reset model back to training mode
    return out                             # return dict with train/eval losses

In [10]:
# create a PyTorch optimizer (AdamW) with given learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) 

for iter in range(max_iters):
    # every eval_iters steps, check train and validation loss
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    # sample a batch of training data
    xb, yb = get_batch('train')
    
    # forward pass: compute logits and loss
    logits, loss = model(xb, yb)
    
    # reset gradients before backprop
    optimizer.zero_grad(set_to_none=True)
    
    # backward pass: compute gradients
    loss.backward()
    
    # update model parameters
    optimizer.step()

step 0: train loss 4.7264, val loss 4.7226
step 200: train loss 3.1178, val loss 3.1245
step 400: train loss 2.6542, val loss 2.6692
step 600: train loss 2.5409, val loss 2.5702
step 800: train loss 2.5200, val loss 2.5259
step 1000: train loss 2.4885, val loss 2.5137
step 1200: train loss 2.4813, val loss 2.5036
step 1400: train loss 2.4863, val loss 2.4990
step 1600: train loss 2.4690, val loss 2.4983
step 1800: train loss 2.4629, val loss 2.4953
step 2000: train loss 2.4607, val loss 2.4875
step 2200: train loss 2.4546, val loss 2.4921
step 2400: train loss 2.4587, val loss 2.4786
step 2600: train loss 2.4560, val loss 2.4860
step 2800: train loss 2.4529, val loss 2.4925


In [11]:
# generate from the model 

# start with context of a single token (0), shape (1,1), on chosen device
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# generate 500 new tokens, decode them back to text, and print result
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


Gofithirro sichas aretctchors l STous.
JUThepouncears ste?
oor mealintimathee ser movis non. h ge R a s, wind ngulffove ou by w na iravak bes thainono hereashavende, t eshin, sk theander y.
Ser anten
Gor t th. lo CKEd, eawhin wed fur prerdy higse lom;
Wher ILUpiek'ty
Tiry bagure med anon:
O, owe y yord bel'd pond,
net ysouFliule.
INGl,
CENToungrrdonil k Bur s thupow ced w.
SEHe, ase ic parl gum hangr t w wane hen seringou INVat? US:COUpoof,
Whred t myeaverentcooshe, whiks d.
D n;
'lsprt.
Wingat!
